# Tool-calling agent

Notebook 08 always hands the LLM the same clauses: notebook 07's top 5 plus three that always apply. Here
the LLM runs as an **agent** that decides what to look up. For each requirement it can search the NASA
clauses with its own queries, check its draft, fix what the checks reject, and then save the review.

| Tool | What it does |
|---|---|
| `retrieve_clause(query)` | Searches the 62 addressable clauses in the Chroma store from notebook 03 |
| `check_requirement(review)` | Runs Lauren's checks in `report_schema.py` on a draft and returns the problems, or "VALID" |
| `log_gap(review)` | Saves the final review. It only accepts a review that passes the checks |

The agent may cite only clauses it has been shown, either in its prompt or from its own searches, so a
citation still can't be made up. The draft format, prompt rules, and checks are the same as notebook 08's
(`review_tools.py`), so the two reports can be compared directly.

**Setup.** Run `03_build_vector_store.ipynb` once to create `data/chroma_db/` (it isn't in git), and put
`GOOGLE_API_KEY=...` in `.env`. Each requirement's final review and tool history are cached in
`data/llm_cache/`, so re-running costs nothing for requirements already reviewed.

In [1]:
import json, sys, time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "rag_retrievals.csv").exists())
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "contract"))
from review_tools import (ALWAYS_INCLUDE, SYSTEM, DraftReview, build_report, clause_block, clauses, fallback,
                          locator, srs, srs_text, to_review, with_api_retries)
from validate_report import validate

load_dotenv(ROOT / ".env")

MODEL = "gemini-3.6-flash"  # pinned: the name notebook 08 asked for ("gemini-3.5-flash") is served as this model
AGENT_VERSION = 1           # bump when the agent's prompt or tools change, so old and new answers are never mixed
CACHE_DIR = ROOT / "data" / "llm_cache" / f"agent_{MODEL}_v{AGENT_VERSION}"
REPORT_PATH = ROOT / "data" / "report_agent.json"
RAG_REPORT_PATH = ROOT / "data" / "report.json"
MAX_STEPS = 30              # LangGraph steps per requirement (each model call or tool call is a step)

## Searching the clauses

The same Chroma store and deduplication as notebook 07: long clauses are split into pieces, so we fetch
extra pieces and keep distinct clauses.

In [2]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = Chroma(
    collection_name="addressable_clauses",
    persist_directory=str(ROOT / "data" / "chroma_db"),
    embedding_function=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"),
)
assert vectorstore._collection.count() > 0, "data/chroma_db is empty: run 03_build_vector_store.ipynb first"

def search_clauses(query, k=5):
    ids = []
    for doc, _ in vectorstore.similarity_search_with_score(query, k=50):
        if doc.metadata["chunk_id"] not in ids:
            ids.append(doc.metadata["chunk_id"])
        if len(ids) == k:
            break
    return ids

print(search_clauses("pilot override of automatic controls"))

['swehb-3.6-guidance', 'swehb-3.2.1.2-guidance', 'swehb-3.2.1.5-guidance', 'swehb-3.2.1.6-guidance', 'swehb-3.2.1.4-guidance']


## Tools

The tools are built fresh for each requirement, so they know which requirement is under review and
which clauses the agent has seen. `log_gap` keeps Mark's original tool name from the project overview.

In [3]:
from langchain_core.tools import tool

def make_tools(row, state):
    """state tracks what the agent saw and saved: {"seen", "queries", "logged"}."""

    def checked(review):
        draft = review if isinstance(review, DraftReview) else DraftReview.model_validate(review)
        return draft, to_review(row, draft, state["seen"])

    @tool
    def retrieve_clause(query: str, k: int = 5) -> str:
        """Search the NASA clauses (NPR 7150.2D and SWE Handbook 5.09) for guidance relevant to `query`.
        Returns up to k clauses, each with the ID to cite in square brackets."""
        ids = search_clauses(query, min(max(k, 1), 8))
        state["seen"].update(ids)
        state["queries"].append(query)
        return clause_block(ids)

    @tool
    def check_requirement(review: DraftReview) -> str:
        """Check a draft review against the output contract. Returns VALID, or the problems to fix."""
        try:
            checked(review)
            return "VALID"
        except ValueError as e:
            return f"NOT VALID. Fix these problems:\n{e}"

    @tool
    def log_gap(review: DraftReview) -> str:
        """Save the final review of the requirement. It must pass check_requirement. Call this once."""
        try:
            draft, _ = checked(review)
        except ValueError as e:
            return f"NOT SAVED. Fix these problems and call log_gap again:\n{e}"
        state["logged"] = draft
        return "SAVED. Reply DONE."

    return [retrieve_clause, check_requirement, log_gap]

## Prompt

The review rules are notebook 08's `SYSTEM`; the agent adds how to use the tools. The three always-included
clauses and the whole SRS are in the first message, so the agent searches for anything more specific.

In [4]:
AGENT_SYSTEM = SYSTEM + """
How to work
1. Use retrieve_clause to find NASA guidance for this requirement. Search more than once with different,
   specific queries when useful: the requirement's topic, a vague or undefined term, a safety concern.
2. Check whether the SRS elsewhere defines the terms the requirement uses, or contains requirements that
   could interact with it.
3. Draft the review and call check_requirement. Fix every problem it reports.
4. Call log_gap once with the final review. After it says SAVED, reply DONE.
Cite only clause IDs you have been shown: the clauses in the first message and those returned by retrieve_clause.
"""

def agent_prompt(row):
    return f"""REQUIREMENT UNDER REVIEW
{row.req_id} ({row.title}) — locator for quotes from it: "{locator(row)}"
{row.text}

NASA CLAUSES THAT ALWAYS APPLY (cite by the ID in square brackets)
{clause_block(ALWAYS_INCLUDE)}

FULL SRS
{srs_text}"""

## Running the agent on one requirement

Each requirement gets a fresh agent. If the agent runs out of steps or never saves a valid review, the
requirement falls back to `UNDETERMINED` and isn't cached, so the next run tries it again.

In [5]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage
from langgraph.errors import GraphRecursionError

llm = init_chat_model(f"google_genai:{MODEL}", max_retries=2)

def run_agent(row):
    state = {"seen": set(ALWAYS_INCLUDE), "queries": [], "logged": None}
    agent = create_agent(llm, make_tools(row, state), system_prompt=AGENT_SYSTEM)
    try:
        result = agent.invoke({"messages": [("human", agent_prompt(row))]}, config={"recursion_limit": MAX_STEPS})
        messages = result["messages"]
    except GraphRecursionError:
        messages = []
    ai = [m for m in messages if isinstance(m, AIMessage)]
    usage = [m.usage_metadata or {} for m in ai]
    stats = {
        "model_calls": len(ai),
        "tool_calls": [c["name"] for m in ai for c in m.tool_calls],
        "input_tokens": sum(u.get("input_tokens", 0) for u in usage),
        "output_tokens": sum(u.get("output_tokens", 0) for u in usage),
    }
    return state, stats

def review_with_agent(row):
    """Returns (review, record). record is None for a fallback; record["cached"] is True if nothing was called."""
    cache = CACHE_DIR / f"{row.req_id}.json"
    if cache.exists():
        record = json.loads(cache.read_text(encoding="utf-8"))
        draft = DraftReview.model_validate(record["review"])
        return to_review(row, draft, record["allowed"]), {**record, "cached": True}

    state, stats = with_api_retries(lambda: run_agent(row))
    if state["logged"] is None:
        print(f"{row.req_id}: no valid review saved within {MAX_STEPS} steps; falling back to UNDETERMINED")
        return fallback(row, f"agent saved no valid review within {MAX_STEPS} steps"), None
    record = {"review": state["logged"].model_dump(), "allowed": sorted(state["seen"]),
              "queries": state["queries"], **stats}
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps(record, indent=2), encoding="utf-8")
    return to_review(row, state["logged"], record["allowed"]), {**record, "cached": False}

## Try a few first

The same four as notebook 08: one good requirement, one with a missing response, and two safety cases.
Check these, and the token use, before running all 54.

In [6]:
TRY_FIRST = ["SFMC-REQ-002", "SFMC-REQ-010", "SFMC-REQ-015", "SFMC-REQ-035"]
records = {}

for _, row in srs[srs.req_id.isin(TRY_FIRST)].iterrows():
    review, records[row.req_id] = review_with_agent(row)
    r = records[row.req_id] or {}
    codes = ", ".join(i.issue_code for i in review.quality_issues) or "-"
    print(f"{review.requirement_id}  {review.quality_status:13} {codes:40} SWE: {review.primary_swe_id or '-'}")
    print(f"    searches: {r.get('queries')}")
    print(f"    {r.get('model_calls')} model calls | tools {r.get('tool_calls')} | "
          f"{r.get('input_tokens')} in / {r.get('output_tokens')} out tokens\n")

SFMC-REQ-002  ACCEPTABLE    -                                        SWE: -
    searches: ['update rate timing requirement display position mode']
    4 model calls | tools ['retrieve_clause', 'check_requirement', 'log_gap'] | 39582 in / 2212 out tokens

SFMC-REQ-010  NEEDS_REVIEW  UNSPECIFIED_RESPONSE                     SWE: SWE-184
    searches: ['unspecified response appropriate corrective action safety control engine failure']
    4 model calls | tools ['retrieve_clause', 'check_requirement', 'log_gap'] | 42932 in / 2935 out tokens

SFMC-REQ-015  NEEDS_REVIEW  UNDEFINED_TERM                           SWE: SWE-184
    searches: ['undefined terms criteria safety constraints inhibits weapons', 'undefined terms testable unambiguous criteria']
    5 model calls | tools ['retrieve_clause', 'retrieve_clause', 'check_requirement', 'log_gap'] | 74332 in / 3411 out tokens

SFMC-REQ-035  NEEDS_REVIEW  SCOPE_AMBIGUITY, UNSPECIFIED_RESPONSE    SWE: SWE-184
    searches: ['override safety const

## Review all 54

In [7]:
reviews = []
for _, row in srs.iterrows():
    review, record = review_with_agent(row)
    records.setdefault(row.req_id, record)  # keep the record from the try-first cell
    reviews.append(review)

done = {k: v for k, v in records.items() if v}
fallbacks = sorted(k for k, v in records.items() if v is None)
usage = pd.DataFrame([{"req_id": k, "searches": len(v["queries"]), "model_calls": v["model_calls"],
                       "input_tokens": v["input_tokens"], "output_tokens": v["output_tokens"]} for k, v in done.items()])
print(len(reviews), "reviewed |", len(fallbacks), "fell back", fallbacks or "")
usage.describe().loc[["mean", "max"]].round(1)

54 reviewed | 0 fell back 


,searches,model_calls,input_tokens,output_tokens
mean,1.4,4.5,54856.0,3296.9
max,2.0,7.0,98594.0,6231.0


## Assemble and validate report_agent.json

Written next to the RAG report rather than over it, so the two can be compared.

In [8]:
limitations = [
    "Requirement reviews only; relationships, document findings, and SWE coverage are not generated yet.",
    "No NASA project compliance is determined from the SRS alone.",
    "Verdicts are LLM-generated and have not been checked by a human reviewer.",
]
if fallbacks:
    limitations.append(f"{len(fallbacks)} requirement(s) failed automated review and are marked UNDETERMINED: {', '.join(fallbacks)}.")

method = (f"Tool-calling agent ({MODEL}) with retrieve_clause, check_requirement, and log_gap: searches the 62 "
          "addressable clauses in Chroma (all-MiniLM-L6-v2) with its own queries, plus 3 always-included clauses; "
          "citations limited to clauses it was shown and filled from addressable_clauses.csv; validated with report_schema.py")
build_report(reviews, method, limitations, REPORT_PATH)
print("validate_report.py counts (reviews, relationships, document findings, SWEs):", validate(REPORT_PATH))

validate_report.py counts (reviews, relationships, document findings, SWEs): (54, 0, 0, 0)


## Compare with the RAG report and the calibration examples

Mark's 7 requirement-level calibration answers, then every requirement where the agent and notebook 08's
RAG pipeline disagree. Nima's evaluation (Task #7) and Task #11 do the full scoring.

In [9]:
calibration = pd.read_csv(ROOT / "data" / "benchmark" / "calibration_examples_for_teams_v1.csv")
codes = lambda r: ", ".join(q.issue_code for q in r["quality_issues"]) if isinstance(r, dict) else ", ".join(q.issue_code for q in r.quality_issues)

agent_by_id = {r.requirement_id: r for r in reviews}
rag_by_id = {r["requirement_id"]: r for r in json.loads(RAG_REPORT_PATH.read_text(encoding="utf-8"))["requirement_reviews"]}

cal = calibration[calibration.scope == "REQUIREMENT"][["target_ids", "expected_outcome", "issue_code"]].copy()
cal["agent"] = cal.target_ids.map(lambda i: agent_by_id[i].quality_status)
cal["agent_codes"] = cal.target_ids.map(lambda i: codes(agent_by_id[i]) or "NONE")
cal["status_ok"] = cal.agent == cal.expected_outcome
cal["code_ok"] = [c in found.split(", ") for c, found in zip(cal.issue_code, cal.agent_codes)]
print(f"calibration: status {cal.status_ok.sum()}/{len(cal)} | issue code {cal.code_ok.sum()}/{len(cal)}")

diff = pd.DataFrame([{
    "req_id": i,
    "rag": f"{rag_by_id[i]['quality_status']} {', '.join(q['issue_code'] for q in rag_by_id[i]['quality_issues'])}",
    "agent": f"{r.quality_status} {codes(r)}",
} for i, r in agent_by_id.items()
  if (r.quality_status, sorted(q.issue_code for q in r.quality_issues))
  != (rag_by_id[i]["quality_status"], sorted(q["issue_code"] for q in rag_by_id[i]["quality_issues"]))])
print(f"agent and RAG differ on {len(diff)} of {len(reviews)} requirements")
diff

calibration: status 7/7 | issue code 6/7
agent and RAG differ on 16 of 54 requirements


,req_id,rag,agent
0,SFMC-REQ-005,NEEDS_REVIEW SCOPE_AMBIGUITY,ACCEPTABLE
1,SFMC-REQ-007,NEEDS_REVIEW VAGUE_CRITERION,"NEEDS_REVIEW VAGUE_CRITERION, SCOPE_AMBIGUITY"
2,SFMC-REQ-011,ACCEPTABLE,NEEDS_REVIEW SCOPE_AMBIGUITY
3,SFMC-REQ-014,ACCEPTABLE,NEEDS_REVIEW UNDEFINED_TERM
4,SFMC-REQ-016,"NEEDS_REVIEW VAGUE_CRITERION, ALLOCATION_UNCLEAR",NEEDS_REVIEW VAGUE_CRITERION
5,SFMC-REQ-020,ACCEPTABLE,"NEEDS_REVIEW UNDEFINED_TERM, SCOPE_AMBIGUITY"
6,SFMC-REQ-026,"NEEDS_REVIEW UNDEFINED_TERM, VAGUE_CRITERION","NEEDS_REVIEW UNDEFINED_TERM, SCOPE_AMBIGUITY"
7,SFMC-REQ-031,NEEDS_REVIEW ALLOCATION_UNCLEAR,ACCEPTABLE
8,SFMC-REQ-032,"NEEDS_REVIEW UNDEFINED_TERM, VAGUE_CRITERION",NEEDS_REVIEW UNDEFINED_TERM
9,SFMC-REQ-035,"NEEDS_REVIEW SCOPE_AMBIGUITY, SAFETY_INTERACTION","NEEDS_REVIEW SCOPE_AMBIGUITY, UNSPECIFIED_RESP..."


## Checks

In [10]:
benchmark = pd.read_csv(ROOT / "data" / "benchmark" / "benchmark_inputs_for_teams_v1.csv")
bench_reqs = set(benchmark.loc[benchmark.scope == "REQUIREMENT", "target_ids"])
assert len(reviews) == len(srs) == 54, "every requirement gets exactly one review"
assert bench_reqs <= set(agent_by_id), f"benchmark requirements missing: {sorted(bench_reqs - set(agent_by_id))}"
assert all(set(v["allowed"]) <= set(clauses.index) for v in done.values()), "only addressable clauses can be cited"
print(f"all {len(bench_reqs)} benchmark requirements reviewed; report_agent.json passes validate_report.py")

all 23 benchmark requirements reviewed; report_agent.json passes validate_report.py
